# MMFDB · Deposition: export → ZIP → re-import

To archive or share a dataset, MMFDB packs it into a **self-contained ZIP** — a
full database snapshot, the provenance graph, a manifest, and the **native raw
data files**. This notebook shows the full round-trip:

1. put data in one MMFDB instance and record its provenance,
2. **export** a deposition ZIP (adding a standards-based **mmCIF** metadata file),
3. **re-import** it into a *second, independent* MMFDB instance and verify nothing
   was lost, and
4. note how to push the ZIP to **Zenodo / OSF** with their own tools.

Because the export is a full snapshot, the second instance is reconstituted from
the ZIP alone — the definition of a portable deposition.

In [1]:
%matplotlib inline
import logging, warnings, hashlib, zipfile, tempfile, json
from pathlib import Path
for _n in ("numexpr", "mmfdb"):
    logging.getLogger(_n).setLevel(logging.WARNING)
warnings.filterwarnings("ignore")

from chisurf.plugins.burst.burst_analysis.api import BurstWorkflow
from mmfdb.repository import MFDatabase
from mmfdb.config import _CONFIG
def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

## 1. Instance A — data with provenance

Simulate a measurement, register it, record a processing operation, and seed the
default fluorophores (so the mmCIF export has probe/spectra content).

In [2]:
A = BurstWorkflow.demo()
client = A._client
sim = A.simulate(fret=(0.25, 0.75), exchange_rate=2.0)
raw = A.fetch(sim.handle)
raw_id, res_id, op_id = f"raw/{sim.handle}", f"bursts/{sim.handle}", f"op/{sim.handle}"

result = raw.parent / "bursts.csv"
result.write_text("proximity_ratio\n0.51\n0.49\n")
client.call("mmfdb.v1.operations.record_with_artifacts", {
    "operation_id": op_id, "operation_type": "burst_selection", "status": "succeeded",
    "software_package": "tttrlib", "software_version": "0.27.0",
    "input_artifacts": [{"artifact_id": raw_id, "artifact_kind": "raw_measurement",
                         "file_path": str(raw), "checksum": sha256(raw), "role": "raw"}],
    "output_artifacts": [{"artifact_id": res_id, "artifact_kind": "burst_table",
                          "file_path": str(result), "checksum": sha256(result), "role": "bursts"}]})
client.call("fluorophores.import_default_set", {})
print("instance A ready — raw:", raw_id)

instance A ready — raw: raw/34dbaec2-fe02-4de9-8936-18d9f9fe82fd


## 2. Export a deposition ZIP

`archive.zip.export` writes a database snapshot, the provenance graph, a manifest,
and (with `include_external_data`) the native raw files. We then add a standards
**mmCIF** metadata file (`metadata.cif`) so the deposition is readable by any
PDBx/mmCIF tool without MMFDB.

In [3]:
zip_path = Path(tempfile.mkdtemp()) / "deposition.zip"
client.call("archive.zip.export", {
    "target_zip_path": str(zip_path), "seed_node_type": "artifact",
    "seed_node_id": raw_id, "include_external_data": True})

# Add a human-readable mmCIF metadata file to the bundle.
with MFDatabase(str(_CONFIG.database_path)) as dbA:
    cif_text = dbA.export_flr_cif_to_text()
with zipfile.ZipFile(zip_path, "a") as z:
    z.writestr("metadata.cif", cif_text)

with zipfile.ZipFile(zip_path) as z:
    contents = z.namelist()
print(f"deposition.zip ({zip_path.stat().st_size/1e3:.0f} kB):")
for name in contents:
    print("   ", name)
print("\nmmCIF metadata (first lines):")
print("\n".join(cif_text.splitlines()[:8]))

deposition.zip (831 kB):
    manifest.json
    database_snapshot.db
    provenance_graph.json
    external_data/bursts.csv
    external_data/simulation.spc
    metadata.cif

mmCIF metadata (first lines):
data_chisurf_flr_export
#
loop_
_flr_fret_analysis.analysis_id
_flr_fret_analysis.experiment_id
_flr_fret_analysis.sample_id
_flr_fret_analysis.type
_flr_fret_analysis.method


## 3. Round-trip — re-import into a second MMFDB instance

The ZIP carries a complete `database_snapshot.db`, so a **separate** instance is
reconstituted from the bundle alone. We extract it to a local folder (the
unpacked deposition) and open the snapshot as instance **B**, then check that its
artifacts, provenance edges, and native raw file all survived.

In [4]:
unpacked = Path(tempfile.mkdtemp())
with zipfile.ZipFile(zip_path) as z:
    z.extractall(unpacked)
print("unpacked deposition folder:")
for p in sorted(unpacked.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(unpacked)}  ({p.stat().st_size/1e3:.0f} kB)")

with MFDatabase(str(unpacked / "database_snapshot.db")) as B:
    artifacts = B.list_artifacts()
    graph = B.export_provenance_graph("artifact", raw_id)
native_raw = list((unpacked / "external_data").glob("*"))

print("\ninstance B (from the ZIP alone):")
print("   artifacts:", sorted(a["artifact_id"] for a in artifacts))
print("   provenance:", len(graph["nodes"]), "nodes,", len(graph["edges"]), "edges")
print("   native raw files:", [p.name for p in native_raw])
assert {raw_id, res_id} <= {a["artifact_id"] for a in artifacts}
assert len(graph["edges"]) >= 2
print("\n\u2713 round-trip verified: instance B reconstituted from the deposition ZIP")

unpacked deposition folder:
   database_snapshot.db  (950 kB)
   external_data/bursts.csv  (0 kB)
   external_data/simulation.spc  (897 kB)
   manifest.json  (1 kB)
   metadata.cif  (1 kB)
   provenance_graph.json  (4 kB)

instance B (from the ZIP alone):
   artifacts: ['bursts/34dbaec2-fe02-4de9-8936-18d9f9fe82fd', 'raw/34dbaec2-fe02-4de9-8936-18d9f9fe82fd']
   provenance: 3 nodes, 2 edges
   native raw files: ['bursts.csv', 'simulation.spc']

✓ round-trip verified: instance B reconstituted from the deposition ZIP


## 4. Depositing to a repository

`deposition.zip` is self-contained, so publishing it is a plain file upload —
use the repositories' own Python tools rather than anything MMFDB-specific:

```bash
# OSF — the osfclient package
pip install osfclient
osf -p <project_id> upload deposition.zip deposition.zip
```

```python
# Zenodo — the REST API (or the zenodo-client package)
import requests
TOKEN = os.environ["ZENODO_TOKEN"]
dep = requests.post("https://zenodo.org/api/deposit/depositions",
                    params={"access_token": TOKEN}, json={}).json()
bucket = dep["links"]["bucket"]
with open("deposition.zip", "rb") as fh:
    requests.put(f"{bucket}/deposition.zip", data=fh, params={"access_token": TOKEN})
```

> **Gap (filed in the project backlog):**
> MMFDB exports a ZIP (`archive.zip.export`) but has **no matching importer** —
> the round-trip above works only because the bundle *is* a full DB snapshot that
> a second instance can adopt wholesale. There is no `archive.zip.import` handler
> to merge a deposition into an existing instance or to restore its object-store
> blobs from `external_data/`.

In [5]:
A.close()
print("Finished.")

Finished.
